# 4. Old spelling, same phrase

In [notebook 3](03-finding-repeated-phrases.ipynb), we found candidate formulas by looking for *exactly* the same sequence of words, repeated across many paragraphs. That works well when a formula is always spelled the same way &mdash; but historical text is rarely that tidy. The same clerk might spell a word two different ways in two different paragraphs; a different clerk, a different scribe's habits, or simply the passage of time can introduce more spelling variation still.

This notebook looks at how the toolkit can recognise that two different spellings are really the same underlying word, and &mdash; just as importantly &mdash; how to tell when it gets this wrong, so you know to check rather than trust it blindly.

In [1]:
import csv
import gzip
import re

TOKEN_PATTERN = re.compile(r"[A-Za-z\u00c0-\u00ff]+")


def tokenize(text):
    return TOKEN_PATTERN.findall(text.lower())


rows = []
with gzip.open('data/resolution_paragraphs/paragraphs-3823.tsv.gz', 'rt', encoding='utf-8') as f:
    for row in csv.DictReader(f, delimiter='\t'):
        rows.append(row)

documents = [tokenize(row['text']) for row in rows]
flat_stream = [token for doc in documents for token in doc]
print(f'{len(documents)} documents, {len(flat_stream)} tokens')

2909 documents, 449905 tokens


## A real example, hiding in plain sight

Dutch has a word for "Lord" or "Mister" that this corpus spells two different ways: `heere` and `heer`. Both are genuinely common &mdash; this isn't a rare typo.

In [2]:
print(f"'heere' occurs {flat_stream.count('heere')} times")
print(f"'heer' occurs {flat_stream.count('heer')} times")

'heere' occurs 1558 times
'heer' occurs 350 times


Are these really the same word, just spelled differently? Let's look at a few examples of each in context, the same way we did with concordances earlier in this series.

In [3]:
def show_examples(word, n=3):
    shown = 0
    for i, token in enumerate(flat_stream):
        if token == word:
            print(' '.join(flat_stream[max(0, i - 4):i + 5]))
            shown += 1
            if shown >= n:
                break


print('Examples of "heere":')
show_examples('heere')
print()
print('Examples of "heer":')
show_examples('heer')

Examples of "heere":
den january praeside den heere van palland tot olinthuys
een missive van den heere grave van degenfeld haar
een missive van den heere lestevenon van berkenroode haar

Examples of "heer":
in plaatse van den heer hendrik smissart weegens de
op den mey den heer gysbert tulleken in plaatse
van gemelde maand den heer eyso de wend in


Both words show up in exactly the same role &mdash; immediately before a person's name, meaning "Lord [Name]" or "Mister [Name]". They're the same word. But to the software from notebook 3, which only compares exact sequences of characters, `heere` and `heer` are two completely unrelated words, as different as `heere` and `table` would be. Any formula that sometimes uses one spelling and sometimes the other will have its evidence needlessly split in two, making it look less consistent &mdash; or less frequent &mdash; than it really is.

## Measuring how similar two spellings are

The toolkit includes a function that estimates how likely two spellings are to be variants of the same word, returning a score between 0 (certainly unrelated) and 1 (identical). It isn't just counting how many letters differ &mdash; it specifically looks for the *kinds* of small changes that are common in historical spelling (a dropped letter at the end of a word, for instance), and weighs those more favourably than other kinds of differences.

In [4]:
from formula_detection.variation.edit import compute_variant_similarity

print('heer vs heere:', compute_variant_similarity('heer', 'heere'))
print('heer vs haar (an unrelated word, meaning "her"/"their"):', compute_variant_similarity('heer', 'haar'))

heer vs heere: 0.75
heer vs haar (an unrelated word, meaning "her"/"their"): 0.5


`heer`/`heere` scores noticeably higher than `heer`/`haar`, even though both pairs differ by roughly the same number of letters &mdash; which is exactly the point of using a purpose-built measure rather than raw letter-counting.

## Doing this automatically, across your whole vocabulary

Checking word pairs one at a time by hand doesn't scale to a vocabulary of thousands of words. `VariantIndex` applies the same kind of similarity check automatically, across every word the software has seen, and builds a map from each spelling to one chosen "canonical" (preferred) spelling &mdash; normally the more frequent of the two, on the assumption that the more common spelling is the more standard one.

How willing it is to group two spellings together is controlled by a threshold: only pairs scoring at or above the threshold get merged. Let's try the default setting first.

In [5]:
from formula_detection.search import FormulaSearch
from formula_detection.variant_index import VariantIndex

formula_search = FormulaSearch(documents, min_term_freq=10, min_cooc_freq=10, skip_size=2)

variant_index = VariantIndex(term_freq=formula_search.term_freq, vocab=formula_search.full_vocab)
variant_index.build()

print('default threshold:', variant_index.sim_threshold)
print('canonical spelling chosen for "heer":', variant_index.canonical('heer'))

1. Iterating over sentences to calculate term frequencies
    full collection size (tokens): 449,905
    full lexicon size (types): 21,185
    minimum term frequency: 10
    minimum frequency lexicon size: 2,880
2. Iterating over sentences to calculate the co-occurrence frequencies


docs: 2,909	num_words: 449,905	num_coocs: 1,125,317	num distinct coocs: 182,121
    co-occurrence index size: 182,121


default threshold: 0.82
canonical spelling chosen for "heer": heer


With the default threshold, `heer` is left as its own word &mdash; not grouped with `heere`. The default is deliberately cautious: `heer`/`heere` scored 0.75, just under the default threshold of 0.82, so nothing happens. Let's lower the threshold and try again.

In [6]:
variant_index_lenient = VariantIndex(term_freq=formula_search.term_freq, vocab=formula_search.full_vocab,
                                     sim_threshold=0.75)
variant_index_lenient.build()

print('lowered threshold:', variant_index_lenient.sim_threshold)
print('canonical spelling chosen for "heer":', variant_index_lenient.canonical('heer'))
print('canonical spelling chosen for "heere":', variant_index_lenient.canonical('heere'))
print('every spelling grouped under', repr(variant_index_lenient.canonical('heere')), ':',
      variant_index_lenient.surface_forms(variant_index_lenient.canonical('heere')))

lowered threshold: 0.75
canonical spelling chosen for "heer": heer
canonical spelling chosen for "heere": heeren
every spelling grouped under 'heeren' : {'keeren', 'heenen', 'heere', 'weeren', 'heerer', 'beeren'}


Two things are worth noticing here, and neither is what we might have naively expected.

**First:** `heere` did get grouped &mdash; but under `heeren` (the *plural*, "Lords"), not directly with `heer`. `heeren` happens to be the more frequent of the lookalike words, so it became the chosen canonical spelling, and `heere` (along with several other similar-looking words) was grouped underneath it.

**Second:** `heer` *still* wasn't grouped with anything, even though we specifically lowered the threshold to let `heer`/`heere` through. When many similar-looking words across an entire vocabulary are being resolved against each other at once, the outcome for one specific pair can be harder to predict than checking that pair in isolation &mdash; in this case, by the time the software got around to comparing `heer`, the word it would have matched (`heere`) had already been claimed by `heeren`.

**And there's a more important problem hiding in the first finding:** `heeren` ("Lords", plural) and `heere` ("Lord", singular) are *not* the same word &mdash; they mean different things. The similarity measure can't tell that apart from spelling alone; it only sees that the two strings look alike. Lowering the threshold caught a real spelling variant we cared about, but it also quietly merged singular and plural into one, which is a mistake for many research purposes.

## The lesson

There is no threshold that's simply "correct". A stricter threshold misses genuine variants (as the default did with `heer`/`heere`); a more lenient one catches more variants but also risks merging words that are related in form but not in meaning (as we just saw with the singular/plural mix-up). Spelling-based matching, on its own, cannot fully solve this &mdash; it has no way to know what a word *means*, only how it's *written*.

This is why `VariantIndex` gives you `canonical()` and `surface_forms()` to inspect exactly what got grouped, rather than asking you to trust the result blindly. **Always check what happened to the specific words that matter for your research question, especially after lowering the threshold.** If spelling-based matching alone isn't precise enough for your material, `VariantIndex` also supports two more advanced techniques &mdash; a rule-based option for known, well-understood historical spelling conventions in a particular language, and a technique that looks at the words *surrounding* a term rather than only its own spelling. Both are more involved to set up and are documented in the toolkit's reference documentation rather than covered here.

## Next steps

So far we've treated each paragraph as a separate, already-known unit of text. But in many real archives, you don't actually know in advance where one document ends and the next begins &mdash; you might only have a long, continuous stream of transcribed text. [Notebook 5](05-where-do-documents-begin-and-end.ipynb) looks at how the formulaic phrases themselves can help find those boundaries.